In [2]:
# Step 2: Import Python Packages: Pandas, Numpy, and Statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.optimize import minimize
import os
from scipy.stats import norm

In [3]:
# Step 3: Set I/O Folders
InputPath = 'inputFolder'

os.makedirs('outputFolder', exist_ok=True)
OutputPath = 'outputFolder'

MIDataFile = '/MIData-Algo-01.csv'

MIResultsFile = '/MIParams_NonLinear.csv'

In [4]:
MIData = pd.read_csv(InputPath + MIDataFile)
MIData

,Size,Volatility,POV,Cost
0,0.076,0.344,0.227,55.9
1,0.746,0.324,0.277,140.1
2,0.390,0.101,0.532,67.0
3,0.025,0.315,0.219,26.3
4,0.227,0.292,0.237,83.3
...,...,...,...,...
295941,0.020,0.425,0.146,23.1
295942,0.013,0.499,0.074,23.2
295943,0.262,0.241,0.214,78.9
295944,0.236,0.159,0.208,59.1


In [5]:
# Step 4. Define Variables

# Load Data File
MIData = pd.read_csv(InputPath +  MIDataFile)

# Define Variables
Cost = MIData['Cost']
Size = MIData['Size']
Volatility = MIData['Volatility']
POV = MIData['POV']

# Set X-independen() variable and Y-dependent variable
X = MIData[['Size', 'Volatility', 'POV']]
Y = MIData['Cost']
n, k = X.shape
    # n = number of observations
    # k = number of independent variables



In [6]:
# Step 5. Define MI Cost Optimization Function
    # We are solving via non-linear least squares
def MI_NonLinear_Obj(MIParams, Size, POV, Volatility, Cost):
    a1, a2, a3, a4, b1 = MIParams
    EstCost = a1 * (Size ** a2) * (Volatility ** a3) * (b1 * (POV ** a4) + (1 - b1))
    Error = Cost - EstCost
    ErrorSq = Error ** 2
    SSE = np.sum(ErrorSq)
    return SSE

def MI_NonLinear_Obj2(initial_guess, Size, POV, Volatility, Cost):
    a1, a2, a3, a4, b1 = initial_guess
    return np.sum(np.square((a1 * (Size ** a2) * (Volatility ** a3) * (b1 * (POV ** a4) + (1 - b1))) - Cost))

In [7]:
# Step 6. Run Non-Linear Regression using Non-Linear Least Squares Optimization

# Initial Guess for Parameters
a1 = 1000
a2 = 0.5
a3 = 0.5
a4 = 0.5
b1 = 0.90
initial_guess = [a1, a2, a3, a4, b1]

# Set Upper and Lower Bounds on Constraints
# LB = [0, 0.0001, 0.0001, 0.0001, 0.0001]
LB = [0, 0, 0, 0, 0]
UB = [2000, 2.0, 2.0, 2.0, 1.0]
MyBounds = zip(LB, UB)

# Rum Non-Linear Regression using scipy.optimize.minimize
results = minimize(MI_NonLinear_Obj, initial_guess, bounds=MyBounds, args=(Size, POV, Volatility, Cost))
print(results)

a1 = results.x[0]
a2 = results.x[1]
a3 = results.x[2]
a4 = results.x[3]
b1 = results.x[4]

print(f"\n----------Estimated Parameters----------\n a1={a1}\n a2={a2}\n a3={a3}\n a4={a4}\n b1={b1}")

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 174906362.30409223
        x: [ 8.477e+02  3.495e-01  7.994e-01  6.476e-01  9.209e-01]
      nit: 27
      jac: [ 0.000e+00 -3.278e+02 -3.010e+02 -2.623e+02 -3.338e+02]
     nfev: 192
     njev: 32
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>

----------Estimated Parameters----------
 a1=847.7300235786264
 a2=0.3495203931438323
 a3=0.7993752553833103
 a4=0.6475891891209415
 b1=0.9208704272714412


In [8]:
# Step 6. Calculate Model Error
EstCost = (a1*Size**a2*Volatility**a3)*(b1*POV**a4 + (1-b1))
ActualCost = Cost

#Calculate IStar Model Error
MSE = np.mean((ActualCost - EstCost) ** 2)
SE = MSE ** 0.5
print(f"----------Model Error----------\n MSE={MSE}\n SE={SE}")

----------Model Error----------
 MSE=591.0076916197287
 SE=24.310649757251014


In [9]:
# Step 7. Save Results to Output Folder

DataResults = [[a1, a2, a3, a4, b1, MSE, SE]]
MIResults = pd.DataFrame(DataResults, columns=['a1', 'a2', 'a3', 'a4', 'b1', 'MSE', 'SE'])
MIResults.to_csv(OutputPath + MIResultsFile, index=False, header=True)
MIResults.head()

,a1,a2,a3,a4,b1,MSE,SE
0,847.730024,0.34952,0.799375,0.647589,0.92087,591.007692,24.31065
